In [2]:
import pandas as pd

DATA_PATH = "../data/clean/mental_econ_inequality_full.csv"
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())

# quick missingness check
missing = df.isna().mean().sort_values(ascending=False)
print("\nMissing ratio (top 15):\n", missing.head(15))

df.head()


Shape: (4556, 12)

Columns:
 ['country', 'Code', 'year', 'depression_rate', 'gdp_per_capita', 'gini_index', 'population', 'gini_index_owid', 'avg_income_usd', 'top10_share', 'bottom10_share', 'income_group']

Missing ratio (top 15):
 avg_income_usd     0.940737
top10_share        0.940737
population         0.940737
gini_index_owid    0.940737
bottom10_share     0.940737
income_group       0.940737
gini_index         0.694908
Code               0.003292
depression_rate    0.000000
year               0.000000
country            0.000000
gdp_per_capita     0.000000
dtype: float64


,country,Code,year,depression_rate,gdp_per_capita,gini_index,population,gini_index_owid,avg_income_usd,top10_share,bottom10_share,income_group
0,Afghanistan,AFG,1990.0,4.071831,963.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,AFG,1991.0,4.079531,881.17,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,1992.0,4.088358,843.88,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,AFG,1993.0,4.096190,578.40,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,AFG,1994.0,4.099582,428.42,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# -----------------------------
# CORE ML DATASET
# -----------------------------

core_features = [
    "gdp_per_capita",
    "gini_index",
    "year"
]

target = "depression_rate"

core_df = df[core_features + [target]].dropna()

print("Core ML shape:", core_df.shape)

X_core = core_df[core_features]
y_core = core_df[target]


Core ML shape: (1390, 4)


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
import pandas as pd

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_core, y_core, test_size=0.2, random_state=42
)

# Train
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

# Predict
y_pred = lin_reg.predict(X_test)

# Evaluate
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Linear Regression R²: {r2:.4f}")
print(f"Linear Regression RMSE: {rmse:.4f}")

Linear Regression R²: 0.1893
Linear Regression RMSE: 0.5720


In [12]:
# Coefficients
coef_df = pd.DataFrame({
    "feature": X_core.columns,
    "coefficient": lin_reg.coef_
}).sort_values(by="coefficient", key=abs, ascending=False)

coef_df


,feature,coefficient
2,year,-0.018337
1,gini_index,-0.008371
0,gdp_per_capita,0.000018


In [13]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

y_pred_ridge = ridge.predict(X_test)

r2_ridge = r2_score(y_test, y_pred_ridge)
rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))

print(f"Ridge Regression R²: {r2_ridge:.4f}")
print(f"Ridge Regression RMSE: {rmse_ridge:.4f}")

ridge_coef_df = pd.DataFrame({
    "feature": X_core.columns,
    "coefficient": ridge.coef_
}).sort_values(by="coefficient", key=abs, ascending=False)

ridge_coef_df


Ridge Regression R²: 0.1893
Ridge Regression RMSE: 0.5720


,feature,coefficient
2,year,-0.018337
1,gini_index,-0.008371
0,gdp_per_capita,0.000018


In [14]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

r2_rf = r2_score(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print(f"Random Forest R²: {r2_rf:.4f}")
print(f"Random Forest RMSE: {rmse_rf:.4f}")

rf_importance = pd.DataFrame({
    "feature": X_core.columns,
    "importance": rf.feature_importances_
}).sort_values(by="importance", ascending=False)

rf_importance


Random Forest R²: 0.5131
Random Forest RMSE: 0.4433


,feature,importance
0,gdp_per_capita,0.558437
1,gini_index,0.305611
2,year,0.135952
